# End-to-End Speech Recognition with Fun-ASR-Nano and OpenVINO

Fun-ASR is an end-to-end speech recognition large model launched by Tongyi Lab. It is trained on tens of millions of hours of real speech data, possessing powerful contextual understanding capabilities and industry adaptability. It supports low-latency real-time transcription and covers 31 languages. It excels in vertical domains such as education and finance, accurately recognizing professional terminology and industry expressions, effectively addressing challenges like "hallucination" generation and language confusion, achieving "clear hearing, understanding meaning, and accurate writing."

<img width="792" height="479" alt="image" src="https://github.com/user-attachments/assets/d55ea91b-0dd2-4a92-b6a1-3460edb41b6f" />


More details can be found in the original [repository](https://github.com/FunAudioLLM/Fun-ASR) and [model card](https://huggingface.co/FunAudioLLM/Fun-ASR-Nano-2512)

In this tutorial we consider how to run and optimize Fun-ASR-Nano using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run Speech Recognition](#Run-Speech-Recognition)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/funasr-nano/funasr-nano.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
# Fetch `notebook_utils` module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("funasr-nano.ipynb")

In [ ]:
from cmd_helper import clone_repo
from pip_helper import pip_install

!pip uninstall -y -q torch torchaudio optimum-intel optimum

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch",
    "nncf",
    "torchaudio",
    "openvino>=2025.4.0",
    "optimum>=1.24.0",
    "optimum-intel>=1.22",
    "transformers>=4.53.0,<5.0.0",
    "funasr>=1.2.7",
    "gradio",
    "huggingface_hub",
)

repo_dir = Path("Fun-ASR")
revision = "efe63c122929bcca095fedc537c3081c5c4ee062"
clone_repo("https://github.com/FunAudioLLM/Fun-ASR.git", revision)

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)


In [3]:
import ipywidgets as widgets

model_ids = ["FunAudioLLM/Fun-ASR-Nano-2512", "FunAudioLLM/Fun-ASR-MLT-Nano-2512"]

model_selector = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Model:",
)


model_selector

Dropdown(description='Model:', options=('FunAudioLLM/Fun-ASR-Nano-2512', 'FunAudioLLM/Fun-ASR-MLT-Nano-2512'),…

The FunASR-Nano pipeline consists of four important parts:

* **Audio Frontend (WavFrontend)**: Extracts audio features (Fbank) from raw waveform input, including frame shift, frame length, and mel-frequency bins configuration.
* **Audio Encoder**: Processes the extracted audio features and converts them into audio embeddings that can be understood by the language model.
* **Text Embeddings**: Converts text tokens into embeddings for the language model.
* **Language Model (LLM)**: A Qwen3-based large language model that takes the merged audio and text embeddings as input and generates transcription output.

The model uses a multimodal approach where audio embeddings are merged with text embeddings before being fed into the LLM for speech-to-text transcription.

In [4]:
from ov_funasr_helper import convert_funasr
from huggingface_hub import snapshot_download

model_dir = Path(model_selector.value.split("/")[-1])
snapshot_download(repo_id=model_selector.value, local_dir=model_dir)

ov_model_dir = Path(model_selector.value.split("/")[-1] + "-ov")
convert_funasr(str(model_dir), ov_model_dir)

/home2/ethan/intel/openvino_notebooks/py_env/lib/python3.10/site-packages/openvino/runtime/__init__.py:10: DeprecationWarning: The `openvino.runtime` module is deprecated and will be removed in the 2026.0 release. Please replace `openvino.runtime` with `openvino`.
  warnings.warn(


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

✅ Fun-ASR-Nano-2512 model already converted. You can find results in Fun-ASR-Nano-2512-ov


PosixPath('Fun-ASR-Nano-2512-ov')

## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

In [5]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["AUTO"])

device

Dropdown(description='Device:', options=('CPU', 'GPU'), value='CPU')

In [6]:
llm_ov_config = {
    "CPU": {},
    "GPU": {"ACTIVATIONS_SCALE_FACTOR": "8.0"},
    "NPU": {
        "ACTIVATIONS_SCALE_FACTOR": "8.0",
        "NPU_USE_NPUW": "YES",
        "NPUW_LLM": "YES",
        "NPUW_ONLINE_PIPELINE": "NONE",
        "MAX_PROMPT_LEN": 1024,
        "NPUW_LLM_MIN_RESPONSE_LEN": 512,
    },
}

In [7]:
from ov_funasr_helper import OVFunASRNano

ov_model = OVFunASRNano(ov_model_dir, device=device.value, llm_ov_config=llm_ov_config[device.value])

✅ Tokenizer loaded from Fun-ASR-Nano-2512-ov
✅ Frontend and inference config loaded from Fun-ASR-Nano-2512-ov/frontend_config.json


To deploy LLM on NPU, you can try

### Run Speech Recognition
[back to top ⬆️](#Table-of-contents:)

In [8]:
wav_path = str(model_dir / "example/en.mp3")
res = ov_model.inference(data_in=[wav_path])
text = res[0][0]["text"]
print(text)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The tribal chieftain called for the boy and presented him with fifty pieces of gold.


## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, model_dir)

# if you are launching remotely, specify server_name and server_port
#  demo.launch(server_name='your server name', server_port='server port in int')
# if you have any issue to launch on your platform, you can pass share=True to launch method:
# demo.launch(share=True)
# it creates a publicly shareable link for the interface. Read more in the docs: https://gradio.app/docs/
try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)